# RAG with Feast Feature Store — Milvus + PostgreSQL + Ray

This notebook demonstrates a **Retrieval Augmented Generation (RAG)** pipeline using:

| Component | Technology |
|-----------|------------|
| **Online Store** | Milvus (vector similarity search) |
| **Offline Store** | PostgreSQL |
| **Registry** | PostgreSQL (SQL) |
| **Compute Engine** | Ray (distributed embedding generation via KubeRay) |
| **Dataset** | HuggingFace `rajpurkar/squad` |

### Prerequisites
- Feast FeatureStore CR deployed and `Ready` (via `setup.sh`)
- Milvus, PostgreSQL, and Ray cluster running in the namespace
- This notebook runs **inside the cluster** (e.g. OpenShift AI workbench) with network access to the Feast services

## 1. Install Dependencies

In [ ]:
%pip install --quiet feast[milvus,ray] sentence-transformers datasets psycopg2-binary

## 2. Read Feature Store Configuration

Load `feature_store.yaml` to inspect how the Feast project is configured — which stores, registry, and compute engine are in use.

In [ ]:
import yaml
from pathlib import Path

FEATURE_REPO_DIR = Path("feature_repo")
fs_yaml_path = FEATURE_REPO_DIR / "feature_store.yaml"

with open(fs_yaml_path) as f:
    fs_config = yaml.safe_load(f)

print("=" * 50)
print("  Feast Feature Store Configuration")
print("=" * 50)
for key, value in fs_config.items():
    if isinstance(value, dict):
        print(f"\n  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")
print("=" * 50)

## 3. Connect to Feature Store & List Registered Features

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=str(FEATURE_REPO_DIR))

print(f"Project: {store.project}")
print(f"\nEntities ({len(store.list_entities())}):")
for entity in store.list_entities():
    print(f"  - {entity.name} (join_keys: {entity.join_keys})")

print(f"\nFeature Views ({len(store.list_all_feature_views())}):")
for fv in store.list_all_feature_views():
    fv_type = type(fv).__name__
    print(f"  - {fv.name} ({fv_type})")
    for field in fv.schema:
        tags = ""
        if hasattr(field, 'vector_index') and field.vector_index:
            tags = f" [vector, dim={field.vector_length}]"
        print(f"      {field.name}: {field.dtype}{tags}")

print(f"\nFeature Services ({len(store.list_feature_services())}):")
for fs in store.list_feature_services():
    print(f"  - {fs.name} (tags: {fs.tags})")

## 4. Prepare Data — Download SQuAD from HuggingFace

Download the SQuAD dataset, deduplicate passages, and save as a parquet file that the `FileSource` references.

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from datasets import load_dataset

dataset = load_dataset("rajpurkar/squad", split="train")

df_raw = dataset.to_pandas()[["title", "context"]].drop_duplicates(subset=["context"]).reset_index(drop=True)
df_raw = df_raw.head(500)  # limit to 500 passages for demo

df_raw["passage_id"] = [f"squad_{i}" for i in range(len(df_raw))]
df_raw["event_timestamp"] = datetime.now(timezone.utc)

data_dir = FEATURE_REPO_DIR / "data"
data_dir.mkdir(exist_ok=True)
parquet_path = data_dir / "squad_passages.parquet"
df_raw.to_parquet(parquet_path, index=False)

print(f"Saved {len(df_raw)} passages to {parquet_path}")
df_raw.head()

## 5. Generate Embeddings

Use `sentence-transformers/all-MiniLM-L6-v2` to generate dense vector embeddings for each passage.

> **Note:** In production, the Ray compute engine would handle this distributedly via the `BatchFeatureView` UDF during `feast materialize`. Here we do it locally for the demo.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(EMBED_MODEL_ID)

texts = df_raw["context"].fillna("").tolist()
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True,
)

print(f"Generated embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

## 6. Write Embeddings to Milvus Online Store

Use `store.write_to_online_store()` to push the passages and their embeddings into Milvus for vector similarity search.

In [ ]:
df_features = df_raw.copy()
df_features["embedding"] = embeddings.tolist()
df_features["embedding_model"] = EMBED_MODEL_ID

print(f"Writing {len(df_features)} passages to Milvus online store...")
store.write_to_online_store(
    feature_view_name="passage_embeddings",
    df=df_features,
)
print("Done! Passages are now searchable via vector similarity.")

## 7. RAG Retrieval — Query Similar Passages

Encode a natural-language query into an embedding, then use Feast's `retrieve_online_documents_v2` to find the most similar passages from Milvus.

In [ ]:
queries = [
    "What is the capital of France?",
    "How does photosynthesis work in plants?",
    "Who wrote the theory of relativity?",
    "What are the largest cities in Europe?",
]

for query in queries:
    query_embedding = model.encode([query], normalize_embeddings=True)[0].tolist()

    results = store.retrieve_online_documents_v2(
        features=[
            "passage_embeddings:embedding",
            "passage_embeddings:title",
            "passage_embeddings:context",
        ],
        query=query_embedding,
        top_k=3,
    )

    df_results = results.to_df()

    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    if not df_results.empty:
        for i, row in df_results.iterrows():
            title = row.get("title", "N/A")
            context = row.get("context", "")
            distance = row.get("distance", "N/A")
            snippet = context[:200] + "..." if len(str(context)) > 200 else context
            print(f"\n  [{i+1}] Title: {title}  |  Score: {distance}")
            print(f"      {snippet}")
    else:
        print("  No results found.")

## 8. Architecture Summary

```
┌──────────────────────────────────────────────────────┐
│                    RAG Pipeline                       │
├──────────────────────────────────────────────────────┤
│                                                      │
│  HuggingFace SQuAD ──► FileSource (parquet)          │
│       │                                              │
│       ▼                                              │
│  Ray Cluster (KubeRay)                               │
│  ┌─────────────────────────────────┐                 │
│  │ BatchFeatureView (mode=ray)     │                 │
│  │ UDF: PassageEmbeddingProcessor  │                 │
│  │   sentence-transformers         │                 │
│  │   all-MiniLM-L6-v2             │                 │
│  └───────────┬─────────────────────┘                 │
│              │                                       │
│       ┌──────┴──────┐                                │
│       ▼             ▼                                │
│  PostgreSQL      Milvus                              │
│  (offline +      (online store)                      │
│   registry)      vector search                       │
│                     │                                │
│                     ▼                                │
│          retrieve_online_documents_v2                │
│          (semantic similarity search)                │
│                     │                                │
│                     ▼                                │
│             Top-K Passages ──► LLM (optional)        │
└──────────────────────────────────────────────────────┘
```